# 06 Pre-trained CNN Architectures (ResNet, VGG, Inception)

## 📚 Learning Objectives

By completing this notebook (~20 min), you will:
- Load **ResNet50**, **VGG16**, and **InceptionV3** from Keras Applications and inspect their structure
- Run a **forward pass** with one of them on a sample image and see output shape
- Understand why we use pre-trained architectures instead of designing from scratch

---

## 🌍 Real life

**Where is this used?** ResNet, VGG, Inception are used in **image search**, **medical imaging**, and **autonomous driving** as backbones for classification or detection.

**In this notebook we use** **pre-trained CNN architectures** (ResNet, VGG, Inception) to **extract features** from images. We use **pre-trained models** (instead of training from scratch) **because** they already learned useful features on ImageNet; we reuse them and save **data and time**.

**📌 Covers slide(s):** **20** — Transfer Learning (VGG, ResNet, fine-tuning). *Do this notebook after that slide.*

---

**Before starting:** Run the imports cell below. First run may download weights.

In [ ]:
# Pretrained CNN Architectures — PyTorch / torchvision
import torch
import torchvision.models as models
import torch.nn as nn
print(f"PyTorch {torch.__version__}")
print("Loading architecture metadata (no weight download)...")

## Theory (short)

- **ResNet:** Residual connections (skip connections); very deep networks; avoids vanishing gradients.
- **VGG:** Stack of 3×3 convs; simple and deep; often used as a baseline.
- **Inception:** Multiple filter sizes in parallel (inception modules); efficient computation.
- **Pre-trained:** Trained on ImageNet (1.2M images, 1000 classes); we use them as **feature extractors** or **fine-tune** for our task.
- **We use pre-trained architectures** instead of training from scratch when we have limited data or similar domain (e.g. natural images).

In [ ]:
# ResNet-50: residual (skip) connections, 50 layers deep
resnet = models.resnet50(pretrained=False)
resnet.eval()
total_params = sum(p.numel() for p in resnet.parameters())
print(f"ResNet50 total parameters: {total_params/1e6:.1f}M")
x = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    out = resnet(x)
print(f"Input: {x.shape}  →  Output: {out.shape}  (1000 ImageNet classes)")
print("Key: skip connections prevent vanishing gradient in deep nets")

## 📥 Inputs & 📤 Outputs

**Inputs:** TensorFlow/Keras, NumPy. We use a **random sample image** (or one from CIFAR/MNIST resized) so we don't need external files.

**Dataset:** Synthetic — random image (no download; used to show pretrained model input/output shape).

**Outputs:** Model summaries (layer count, params), output shape after forward pass, and a short comparison sentence.

In [ ]:
# VGG-16: very deep stack of 3×3 convolutions, simple baseline
vgg = models.vgg16(pretrained=False)
vgg.eval()
vgg_params = sum(p.numel() for p in vgg.parameters())
print(f"VGG16 total parameters: {vgg_params/1e6:.1f}M")
with torch.no_grad():
    out_vgg = vgg(x)
print(f"Output: {out_vgg.shape}  (1000 ImageNet classes)")
print("Key: large parameter count due to fully-connected layers")

## Step 1: Imports

In [ ]:
# MobileNetV2: lightweight depthwise-separable convolutions for edge devices
mobile = models.mobilenet_v2(pretrained=False)
mobile.eval()
mob_params = sum(p.numel() for p in mobile.parameters())
print(f"MobileNetV2 total parameters: {mob_params/1e6:.1f}M")
with torch.no_grad():
    out_mob = mobile(x)
print(f"Output: {out_mob.shape}  (1000 ImageNet classes)")
print("Key: ~25× fewer params than ResNet50 — perfect for mobile/edge")

## Step 2: Load ResNet50 (we use pre-trained ResNet instead of training from scratch to reuse ImageNet features)

In [ ]:
# EfficientNet-B0: NAS-optimized compound scaling
# (available in torchvision >= 0.11)
try:
    effnet = models.efficientnet_b0(pretrained=False)
    eff_params = sum(p.numel() for p in effnet.parameters()) / 1e6
except AttributeError:
    eff_params = 5.3  # approximate if old torchvision
print(f"EfficientNetB0 total parameters: {eff_params:.1f}M")
print("Key: compound scaling of depth/width/resolution — best accuracy/param tradeoff")

## Step 3: Compare with VGG16 and InceptionV3 (same idea: pre-trained feature extractors)

In [ ]:
# Visualization: Parameter count comparison
import matplotlib.pyplot as plt

resnet_p  = sum(p.numel() for p in resnet.parameters()) / 1e6
vgg_p     = sum(p.numel() for p in vgg.parameters()) / 1e6
mobile_p  = sum(p.numel() for p in mobile.parameters()) / 1e6
try:
    eff_p = sum(p.numel() for p in effnet.parameters()) / 1e6
except:
    eff_p = 5.3

models_names = ['VGG16', 'ResNet50', 'EfficientNetB0', 'MobileNetV2']
params_M     = [vgg_p, resnet_p, eff_p, mobile_p]
colors = ['#c0392b', '#2980b9', '#f39c12', '#27ae60']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(models_names, params_M, color=colors, edgecolor='black', width=0.5)
for bar, p in zip(bars, params_M):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
            f'{p:.1f}M', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_title('Pretrained CNN Architectures: Parameter Count Comparison', fontsize=13, fontweight='bold')
ax.set_ylabel('Parameters (Millions)')
ax.set_xlabel('Architecture')
plt.tight_layout()
plt.show()
print("Takeaway: VGG is large but simple; MobileNet & EfficientNet are lean and efficient")

## Step 4: Forward pass on a sample image (we use ResNet to get feature logits for one image)

## 🌍 Real-World Worked Example — Fine-Tune ResNet on Custom Categories

**Industry context:**
- Google Photos uses transfer learning to classify your personal photos  
- Hospitals fine-tune ImageNet models on their X-ray datasets with <1000 images
- E-commerce platforms fine-tune ResNet to identify product defects

We fine-tune a **pretrained ResNet-18** (ImageNet weights) on a small binary classification task.

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import torchvision, torchvision.transforms as T
from torch.utils.data import DataLoader, Subset

# ── Use CIFAR-10 classes 0 (airplane) vs 1 (automobile) as our 'custom' data
transform = T.Compose([
    T.Resize(64), T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])  # ImageNet stats
])
full = torchvision.datasets.CIFAR10('/tmp/cifar10', train=True, download=True, transform=transform)
# Keep only classes 0 and 1
idx = [i for i,(x,y) in enumerate(full) if y in (0,1)][:400]
subset = Subset(full, idx)
train_size = int(0.8*len(subset))
train_ds, val_ds = torch.utils.data.random_split(subset, [train_size, len(subset)-train_size])
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=32)

# ── Load pretrained ResNet-18, replace final layer ──────────────────────────
model = torchvision.models.resnet18(weights='IMAGENET1K_V1')
for p in model.parameters(): p.requires_grad = False        # Freeze backbone
model.fc = nn.Linear(model.fc.in_features, 2)               # Only train head

opt     = optim.Adam(model.fc.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(5):
    model.train(); total_loss=0
    for X,y in train_dl:
        y_bin = (y % 2)  # remap to 0/1
        loss = loss_fn(model(X), y_bin)
        opt.zero_grad(); loss.backward(); opt.step()
        total_loss += loss.item()
    model.eval(); correct=0; total=0
    with torch.no_grad():
        for X,y in val_dl:
            y_bin = (y%2)
            correct += (model(X).argmax(1)==y_bin).sum().item(); total+=len(y_bin)
    print(f"Epoch {epoch+1}/5 — loss: {total_loss/len(train_dl):.3f} | val acc: {correct/total*100:.1f}%")

print("\n✅ With only 400 images and 5 epochs, transfer learning gives strong results.")
print("A model trained from scratch would need 100x more data for similar performance.")

## 🧩 Mini-exercise

**Try it:** In a new cell, load VGG16 and run a forward pass on the same sample image you used for ResNet. Compare the output shape and (if you printed it) the prediction. Or print the names of the last three layers of ResNet50.

---

## ✅ Summary

**What you did:** Loaded ResNet50, VGG16, InceptionV3; compared param counts; ran a forward pass with ResNet on a sample image.

**In real life you'd also:** Replace the top layer for your classes, freeze base and train head, or fine-tune last layers.

**The main idea:** Pre-trained CNNs (ResNet, VGG, Inception) give strong feature extractors; we use them instead of training from scratch to save data and time.

**Next:** `05_transfer_learning_cnns` freezes the base and trains a new head; `07_training_cnn_image_datasets` trains a CNN on CIFAR-10.

## 📚 References & Further Reading

**Papers:**
- Tan et al. (2019) — [EfficientNet](https://arxiv.org/abs/1905.11946)
- Dosovitskiy et al. (2020) — [ViT: Vision Transformer](https://arxiv.org/abs/2010.11929)
- He et al. (2016) — [ResNet](https://arxiv.org/abs/1512.03385)

**Practical Guide:** [torchvision Transfer Learning Tutorial](https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)

**State-of-the-Art:** In 2025, fine-tuning a pretrained ViT-L on 100 medical images achieves radiologist-level performance.